# spec

> Any documented HTTP API as a tool group: load a spec, read the operations, call one.

In [ ]:
#| default_exp spec

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from pathlib import Path
from fastcore.test import test_eq, test_fail

`LocalHost` can fetch pages and run commands, but neither operation describes an API. Without a specification, an agent must guess request paths and fields. A wrong request can return a plausible `200` response or an unhelpful `400`.

`fastspec` parses OpenAPI, Google discovery documents and GraphQL introspection into `OpSpec` records. Each record contains the verb, path, parameters, types, defaults and documentation. `fastcore.apisurface` converts those records into Python signatures such as `list_repo_issues(owner, repo, state='open')`.

## Why use the specification instead of a wrapper

Use a maintained wrapper when one exists. Specifications cover internal services, stale clients and one-off endpoints. `fossick` finds specifications. `apis` captures API calls from a page. `fetch` reads documentation pages.

## Why the host owns specifications

A loaded specification remains available for the session. The agent can browse its operations without fetching it again. Keeping this state on the host gives the CLI and embedded IDEs one implementation.

In [ ]:
#| export
import json
from pathlib import Path
from urllib.parse import urlparse

from fastcore.meta import delegates
from ramabana.core import AgentError, agent_err
from ramabana.tools import LocalHost

In [ ]:
#| export
MAX_OPS = 400          # a spec larger than this is a catalogue, not a working surface
class SpecError(AgentError): "A specification could not be read, or an operation could not be called."

def parse_spec(text, why=''):
    """A spec document as a dict, whether it is JSON or YAML.
    OpenAPI is published as both, and more often as YAML. Reading only JSON meant the common case
    failed inside `json.loads` with a character offset, which tells nobody to convert anything.
    """
    text = str(text or '')
    if text.lstrip()[:1] == '<':
        raise SpecError(f'{why or "that address"} served a web page, not a specification. '
                        'Link to the raw file: on GitHub that is the "Raw" button, or '
                        'raw.githubusercontent.com in place of github.com/.../blob.')
    try: return json.loads(text)
    except Exception as je:
        try: import yaml
        except ImportError:
            raise SpecError(f'{why or "the spec"} is not JSON, and PyYAML is not installed to try '
                            f'YAML: {agent_err(je)}') from je
        try: d = yaml.safe_load(text)
        except Exception as ye:
            raise SpecError(f'{why or "the spec"} parsed as neither JSON nor YAML: {agent_err(ye)}') from ye
    if not isinstance(d, dict): raise SpecError(f'{why or "the spec"} is not a specification document')
    return d

def raw_url(s):
    """The raw file behind a code-host *page* url.

    `github.com/o/r/blob/main/openapi.json` is a web page with the file rendered inside it, and
    fetching it gets HTML. It is also the url anyone actually has, being the one in the address bar.
    """
    u = urlparse(str(s or ''))
    parts = [p for p in u.path.split('/') if p]
    if u.netloc.removeprefix('www.') == 'github.com' and len(parts) > 4 and parts[2] in ('blob', 'raw'):
        return f"https://raw.githubusercontent.com/{parts[0]}/{parts[1]}/{'/'.join(parts[3:])}"
    if u.netloc.removeprefix('www.') == 'gitlab.com' and '/-/blob/' in u.path:
        return f'https://gitlab.com{u.path.replace("/-/blob/", "/-/raw/", 1)}'
    if u.netloc.removeprefix('www.') == 'bitbucket.org' and '/src/' in u.path:
        return f'https://bitbucket.org{u.path.replace("/src/", "/raw/", 1)}'
    return str(s or '')

def load_spec(src, timeout=30):
    "A spec from a URL, a path, or an already-parsed dict, routed on what `src` is. JSON or YAML."
    if isinstance(src, dict): return src
    s = str(src or '').strip()
    if not s: raise SpecError('a spec url, path or dict is required')
    if urlparse(s).scheme in ('http', 'https'):
        s = raw_url(s)
        # Through fossick, which is this agent's web layer everywhere else: it carries the headers
        # a bare client does not, and a plain `httpx.get` fails outright behind some TLS proxies.
        from fossick import get_page
        try: r = get_page(s, timeout=timeout)
        except Exception as e: raise SpecError(f'could not read the spec at {s}: {agent_err(e)}') from e
        if r.status != 200: raise SpecError(f'{s} answered {r.status}')
        return parse_spec((r.body or b'').decode('utf-8', errors='replace'), s)
    p = Path(s).expanduser()
    if not p.is_file(): raise SpecError(f'no spec at {p}')
    return parse_spec(p.read_text(), str(p))

def norm_paths(spec):
    """`spec` with `x-ms-paths` folded into `paths`.
    Swagger 2.0 cannot express two operations that differ only by query string. Azure puts
    those under `x-ms-paths` instead. Azure Blob Storage declares sixty of them and leaves `paths`
    empty, which read as a spec with no operations at all. The whole of Azure's data plane
    looked like an empty document. `paths` wins a collision, being the standard key.
    """
    extra = spec.get('x-ms-paths') or {}
    if not extra: return spec
    return {**spec, 'paths': {**extra, **(spec.get('paths') or {})}}

def spec_ops(spec):
    "Every operation in `spec` as `OpSpec` records, whatever flavour of spec it is."
    return parse_ops(spec).ops[:MAX_OPS]

def op_row(op):
    "One operation as the shape a person or a model reads, signed by `fastcore.apisurface`."
    from fastcore.apisurface import mk_sig, sanitized_params
    try: sig = str(mk_sig(op, sanitized_params(_op_params(op)), op.param_defaults))
    except Exception: sig = '(...)'
    return dict(group=op.group or '', name=op.name, verb=(op.verb or '').upper(), path=op.path,
                summary=(op.summary or '').strip(), signature=f'{op.name}{sig}',
                required=list(op.required_params or []), docs_url=op.docs_url or '')

def _op_params(op):
    "Every parameter name an operation takes, in the order the signature wants them."
    return [*(op.route_params or []), *(op.query_params or []),
            *(op.body_params or []), *(op.file_params or [])]

In [ ]:
#| export
def parse_ops(spec):
    "A `fastspec.SpecParser` for an OpenAPI/Azure or Google Discovery document."
    from dataclasses import replace
    from fastcore.basics import AttrDict
    from fastspec.spec import SpecParser
    try:
        parsed = (SpecParser.from_discovery(spec) if 'discoveryVersion' in spec
                  else SpecParser.from_openapi(AttrDict(norm_paths(spec))))
    except Exception as e: raise SpecError(f'could not read the operations: {agent_err(e)}') from e
    if not parsed.ops: raise SpecError('the spec declares no operations')
    parsed.ops = [replace(op, group='/'.join(op.group)) if isinstance(op.group, (list, tuple)) else op
                  for op in parsed.ops]
    return parsed

Azure Blob Storage demonstrates a format that a small example would miss. Its Swagger 2.0 document declares sixty operations under `x-ms-paths` and leaves `paths` empty. Swagger 2.0 cannot put operations that differ only by query string under the same path. A parser that reads only `paths` therefore finds no operations.

`nbs/fixtures/azure-blob-excerpt.json` contains three original entries. The example needs no network access.

In [ ]:
azure = load_spec(Path('fixtures/azure-blob-excerpt.json'))
assert azure['swagger'] == '2.0' and not azure['paths'], 'the whole point: paths is empty'
assert len(azure['x-ms-paths']) == 3

ops = spec_ops(azure)
rows = [op_row(o) for o in ops]
assert len(rows) == 6, 'three paths, six operations, none of them under `paths`'
assert {r['verb'] for r in rows} == {'GET', 'PUT', 'DELETE'}
assert any(r['path'] == '/?comp=list' for r in rows), 'a query-only path survived'
assert all(r['signature'].startswith(r['name']) for r in rows)

# `paths` wins a collision, being the standard key.
clash = dict(azure, paths={'/?comp=list': {'get': {'operationId': 'mine'}}})
assert norm_paths(clash)['paths']['/?comp=list'] == {'get': {'operationId': 'mine'}}

for r in rows[:3]: print(f"{r['verb']:6} {r['path'][:38]:40} {r['name']}")

PUT    /?restype=service&comp=properties        service__set_properties
GET    /?restype=service&comp=properties        service__get_properties
GET    /?comp=list                              service__list_containers_segment


In [ ]:
# The url anyone has is the page url, and the page is HTML. It is rewritten to the file.
test_eq(raw_url('https://github.com/stripe/openapi/blob/master/openapi/spec3.json'),
        'https://raw.githubusercontent.com/stripe/openapi/master/openapi/spec3.json')
test_eq(raw_url('https://github.com/o/r/blob/main/a/b.yaml?plain=1#L4'),
        'https://raw.githubusercontent.com/o/r/main/a/b.yaml')
test_eq(raw_url('https://gitlab.com/o/r/-/blob/main/api.yaml'), 'https://gitlab.com/o/r/-/raw/main/api.yaml')
# anything else is left exactly as it was
for u in ('https://raw.githubusercontent.com/o/r/main/s.json', 'https://api.example.com/openapi.json',
          'https://github.com/o/r', 'specs/local.yaml'):
    test_eq(raw_url(u), u)

# and when a page is what comes back anyway, say that, rather than reporting a yaml scanner error
test_fail(lambda: parse_spec('<!DOCTYPE html><html><body>hi</body></html>', 'that address'),
          contains='served a web page')

In [ ]:
#| export
class SpecHost(LocalHost):
    "`LocalHost` plus named API specs (OpenAPI / Discovery / GraphQL) loaded on demand."

    @delegates(LocalHost.__init__)
    def __init__(self,
                 roots=('.',),          # the folders the agent is confined to
                 specs=None,            # name -> parsed spec, for a host that starts loaded
                 headers=None,          # sent with every API call, whichever spec it is
                 creds=None,            # name -> headers, sent only with calls to that spec
                 timeout=60.0,          # per-call timeout, in seconds
                 max_ops=MAX_OPS,       # rows one `api_ops` answers with; 0 for every one
                 **kwargs):             # forwarded to `LocalHost`
        super().__init__(roots, **kwargs)
        self.specs, self.headers, self.timeout = dict(specs or {}), dict(headers or {}), timeout
        self._creds = {str(k): dict(v) for k, v in (creds or {}).items()}
        self.max_ops = max_ops
        self._clients = {}
        self.spec_info = {}
        # this class answers the api group itself, without an `apis` backend handed in
        self.without = self.without - {'api'}

    def api_load(self, src, name=''):
        "Read a spec and remember it under `name` (default: its title, else the host)."
        spec = load_spec(src)
        parsed = parse_ops(spec)
        info = spec.get('info') or {}
        key = str(name or info.get('title') or spec.get('title') or urlparse(str(src)).netloc or 'api').strip()
        self.specs[key] = parsed
        self._clients.pop(key, None)
        groups = sorted({o.group or '' for o in parsed.ops})
        total, shown = len(parsed.ops), len(self._page(parsed.ops))
        out = dict(name=key, title=info.get('title', spec.get('title', '')),
                   version=info.get('version', spec.get('version', '')),
                   operations=total, groups=groups)
        if shown < total:
            out |= dict(shown=shown, truncated=True,
                        more=f'{total - shown} more: narrow with `group` or `match`, or page with '
                             f'`limit` and `offset`')
        self.spec_info[key] = {**out, 'url': src if isinstance(src, str) else ''}
        return out

    def api_names(self): return sorted(self.specs)

    def _spec(self, name=''):
        if not self.specs: raise SpecError('no specification loaded; call api_load first')
        if not name:
            if len(self.specs) > 1:
                raise SpecError(f'name which api: {", ".join(sorted(self.specs))}')
            name = next(iter(self.specs))
        if name not in self.specs: raise SpecError(f'no api {name!r}; loaded: {", ".join(sorted(self.specs))}')
        return name, self.specs[name]

    def _page(self, rows, limit=None, offset=0):
        "One page of `rows`. `limit=0` is every one. `None` takes the host's own default."
        n = self.max_ops if limit is None else limit
        offset = max(0, int(offset or 0))
        return rows[offset:] if not n else rows[offset:offset + int(n)]

    def api_ops(self, group='', name='', match='', limit=None, offset=0):
        "Operations, narrowed by group or by a substring of the name or summary, then paged."
        _, parsed = self._spec(name)
        rows = [op_row(o) for o in parsed.ops]
        if group: rows = [r for r in rows if r['group'] == group]
        if match:
            m = match.lower()
            rows = [r for r in rows if m in r['name'].lower() or m in r['summary'].lower()]
        return self._page(rows, limit, offset)

    def api_count(self, group='', name='', match=''):
        "How many operations that narrowing matches, whatever a page of it holds."
        return len(self.api_ops(group=group, name=name, match=match, limit=0))

    def api_creds(self, name, headers=None):
        "Set what is sent with calls to one spec and nothing else. `headers=None` clears it."
        if headers: self._creds[str(name)] = dict(headers)
        else: self._creds.pop(str(name), None)
        self._clients.pop(str(name), None)   # a built client captured the old headers
        return sorted(self._creds)

    def api_keyed(self):
        "Which specs carry credentials, and which headers they send — never the values."
        return {k: sorted(v) for k, v in self._creds.items()}

    def _client(self, key, parsed):
        "One client per spec, built with that spec's own credentials over the host's headers."
        if key not in self._clients:
            from fastspec.oapi import OpenAPIClient
            self._clients[key] = OpenAPIClient(parsed, timeout=self.timeout, sync=True,
                                               headers={**self.headers, **self._creds.get(key, {})})
        return self._clients[key]

    def api_call(self, operation, name='', **params):
        """Call one operation by name, with its own parameter names.

        Sync on purpose: a tool call is a blocking step in a turn, and an async client here
        would mean every caller managing a loop to get one response.
        """
        key, parsed = self._spec(name)
        client = self._client(key, parsed)
        fn = getattr(client, str(operation), None)
        if not callable(fn): fn = _in_groups(client, str(operation))
        if fn is None: raise SpecError(f'{key} declares no operation {operation!r}')
        try: return fn(**params)
        except Exception as e: raise SpecError(f'{operation} failed: {agent_err(e)}') from e

def _in_groups(client, operation):
    "The operation on whichever group holds it, for a client that files its ops by path segment."
    for attr in dir(client):
        if attr.startswith('_'): continue
        try: fn = getattr(getattr(client, attr), operation, None)
        except Exception: continue
        if callable(fn): return fn

In [ ]:
#| export
def _head(s):
    "A heading: no underscores, first character uppercase."
    s = ' '.join(str(s or '').replace('_', ' ').replace('-', ' ').split())
    return s[:1].upper() + s[1:]

def op_heading(r):
    "`POST /widgets/{id}` — unique per operation, and readable without the spec beside you."
    return f"{str(r.get('verb') or 'GET').upper()} {r.get('path') or r.get('name') or ''}".strip()

def op_markdown(r):
    "One operation: what it does, how to call it, and what it insists on."
    lines = [f'### {op_heading(r)}', '']
    if r.get('summary'): lines += [r['summary'], '']
    if r.get('name'): lines += [f"Call it as `{r.get('signature') or r['name']}`.", '']
    if r.get('required'): lines += [f"Required: {', '.join(r['required'])}", '']
    if r.get('docs_url'): lines += [f"Docs: {r['docs_url']}", '']
    return '\n'.join(lines)

def spec_markdown(host, name=''):
    """A loaded specification as markdown: one `##` per group, one `###` per operation.
    Returns `(title, markdown)`. Headings start with an uppercase character on purpose: every name
    in a spec is lowercase, and readers that infer structure from markdown commonly treat a
    lowercase `#` line as punctuation rather than a heading.
    """
    info = dict(getattr(host, 'spec_info', {}).get(name) or {})
    # `limit=0`: a document is the one place the whole catalogue belongs. The page size is for
    # a turn, which this is not.
    rows = sorted(host.api_ops(name=name, limit=0), key=lambda r: (r.get('group') or '', r.get('name') or ''))
    title = info.get('title') or _head(name) or 'API'
    version = f" in version {info['version']}" if info.get('version') else ''
    src = info.get('url') or name
    out = [f"{len(rows)} operations{version}" + (f", from `{src}`." if src else '.'), '']
    for group in dict.fromkeys(r.get('group') for r in rows):
        out += [f"## {_head(group) or 'Operations'}", '']
        out += [op_markdown(r) for r in rows if r.get('group') == group]
    return title, '\n'.join(out)

## Reading a spec

The following examples load and call specifications without network access.

In [ ]:
SPEC = {'openapi': '3.0.0', 'info': {'title': 'Widgets', 'version': '1.2.0'},
        'paths': {'/widgets': {'get': {'operationId': 'listWidgets', 'tags': ['widgets'],
                                       'summary': 'Every widget',
                                       'parameters': [{'name': 'limit', 'in': 'query',
                                                       'schema': {'type': 'integer'}}]}},
                  '/widgets/{id}': {'get': {'operationId': 'getWidget', 'tags': ['widgets'],
                                            'summary': 'One widget',
                                            'parameters': [{'name': 'id', 'in': 'path',
                                                            'required': True,
                                                            'schema': {'type': 'string'}}]}}}}

h = SpecHost(roots=['.'])
loaded = h.api_load(SPEC, 'widgets')
test_eq(loaded['operations'], 2)
test_eq(loaded['title'], 'Widgets')

rows = {r['name']: r for r in h.api_ops(name='widgets')}
test_eq(sorted(rows), ['get_widget', 'list_widgets'])
# the signature carries the service's own parameter names, which is the whole point
assert 'id' in rows['get_widget']['signature'], rows['get_widget']['signature']
assert 'limit' in rows['list_widgets']['signature'], rows['list_widgets']['signature']
test_eq(rows['get_widget']['verb'], 'GET')
test_eq(rows['get_widget']['required'], ['id'])

# narrowing, because a real spec has hundreds of these
test_eq([r['name'] for r in h.api_ops(match='one widget', name='widgets')], ['get_widget'])
test_eq(h.api_ops(group='nothing', name='widgets'), [])

# naming which api is required once more than one is loaded
h.api_load(SPEC, 'other')
test_fail(lambda: h.api_ops(), contains='name which api')
test_fail(lambda: h.api_ops(name='nope'), contains='no api')
print(rows['get_widget']['signature'])

get_widget(id: str)


### A specification as a document

A real specification can contain hundreds of operations. `spec_markdown` renders the loaded operations as reference prose instead of returning one large row list. Each group gets a `##` heading. Each operation gets a `###` heading, its purpose, its call signature and its requirements. The renderer uses the same rows as `api_ops`.

Specification names are commonly lowercase. Some Markdown readers do not recognize lowercase headings reliably. The renderer title-cases group headings and identifies each operation by verb and path. The pair remains unique when operations share a path.

`SpecParser` retains only `base_url` and `ops`. `api_load` records the title, version and source in `spec_info` before discarding the parser metadata.

In [ ]:
# A real spec is a catalogue: one page by default, every one on request, and it says so.
BIG = {'openapi': '3.0.0', 'info': {'title': 'Big', 'version': '1'},
       'paths': {f'/g{i//10}/thing{i}': {'get': {'operationId': f'getThing{i}',
                                                 'summary': 'the last one' if i == 24 else f'thing {i}'}}
                 for i in range(25)}}
small = SpecHost(roots=['.'], max_ops=10)
info = small.api_load(BIG, 'big')
test_eq(info['operations'], 25)              # what the spec declares, not what a page holds
test_eq(info['shown'], 10)
assert info['truncated'] and 'limit' in info['more'], info
test_eq(len(small.api_ops(name='big')), 10)
test_eq(len(small.api_ops(name='big', limit=0)), 25)          # 0 is every one
test_eq(len(small.api_ops(name='big', limit=5, offset=22)), 3)
test_eq(small.api_count(name='big'), 25)
# groups come from every operation, including those past the page
test_eq(info['groups'], ['g0', 'g1', 'g2'])

# narrowing happens before paging: the match that only exists on the last operation is found
test_eq([r['name'] for r in small.api_ops(name='big', match='the last one')], ['get_thing24'])
test_eq(small.api_count(name='big', group='g2'), 5)

# a host that was given no cap answers with everything, and says nothing about truncation
whole = SpecHost(roots=['.'], max_ops=0)
assert 'truncated' not in whole.api_load(BIG, 'big')
test_eq(len(whole.api_ops(name='big')), 25)

In [ ]:
ADMIN = {'openapi': '3.0.0', 'info': {'title': 'Widgets', 'version': '1.2.0'},
         'paths': {**SPEC['paths'],
                   # the group is the first path segment, not the `tags` entry
                   '/admin/flush': {'post': {'operationId': 'flush', 'tags': ['widgets'],
                                             'summary': 'Empty the cache'}}}}
d = SpecHost(roots=['.'])
d.api_load(ADMIN, 'widgets')

assert not getattr(d.specs['widgets'], 'title', None)
test_eq(d.spec_info['widgets']['title'], 'Widgets')
test_eq(d.spec_info['widgets']['version'], '1.2.0')
test_eq(d.spec_info['widgets']['url'], '')
test_eq(SpecHost(roots=['.']).spec_info, {})

title, doc = spec_markdown(d, 'widgets')
test_eq(title, 'Widgets')
assert doc.startswith('3 operations in version 1.2.0'), doc[:60]
# Groups and operations use stable heading order.
assert '## Admin' in doc and '## Widgets' in doc
assert doc.index('## Admin') < doc.index('## Widgets')
assert '### GET /widgets/{id}' in doc and '### POST /admin/flush' in doc
assert 'Call it as `get_widget(' in doc and 'Required: id' in doc
# The caller renders the separately returned title.
assert not doc.startswith('#') and '\n# ' not in doc

heads = [l for l in doc.splitlines() if l.startswith('#')]
assert heads and all(l.lstrip('#').strip()[0].isupper() for l in heads), heads

# Rendering works without `spec_info`.
d.spec_info.clear()
test_eq(spec_markdown(d, 'widgets')[0], 'Widgets')

test_eq(op_heading({'verb': 'delete', 'path': '/widgets/{id}'}), 'DELETE /widgets/{id}')
bare = op_markdown({'verb': 'get', 'path': '/x', 'name': 'x'})
assert bare.startswith('### GET /x') and 'Required' not in bare and 'Docs' not in bare


d.api_load(ADMIN, 'second')
test_fail(lambda: spec_markdown(d, ''), contains='name which api')
print(doc)

# Documentation includes every operation despite API pagination.
paged = SpecHost(roots=['.'], max_ops=1)
paged.api_load(ADMIN, 'widgets')
assert spec_markdown(paged, 'widgets')[1].count('###') == 3

In [ ]:
# The group appears because the host declares `api`, and stays absent on a plain LocalHost.
from ramabana.tools import tools_for
names = {t.__name__ for t in tools_for(SpecHost(roots=['.']))}
assert {'api_load', 'api_ops', 'api_call'} <= names, sorted(names)
assert not {'api_load', 'api_ops', 'api_call'} & {t.__name__ for t in tools_for(LocalHost(roots=['.']))}
print(sorted(n for n in names if n.startswith('api_')))

['api_call', 'api_load', 'api_ops']


In [ ]:
# A Google Discovery document uses a different operation tree from OpenAPI, but loads and calls through the same parsed shape.
GCP = {
    'kind': 'discovery#restDescription', 'discoveryVersion': 'v1', 'name': 'storage', 'version': 'v1',
    'title': 'Cloud Storage JSON API', 'baseUrl': 'https://storage.googleapis.com/storage/v1/',
    'resources': {'buckets': {'methods': {'get': {
        'id': 'storage.buckets.get', 'path': 'b/{bucket}', 'httpMethod': 'GET',
        'parameters': {'bucket': {'location': 'path', 'required': True, 'type': 'string'},
                       'projection': {'location': 'query', 'type': 'string'}},
        'description': 'Returns metadata for the specified bucket.'
    }}}}
}
gcp = SpecHost(roots=['.'])
loaded_gcp = gcp.api_load(GCP, 'gcp')
test_eq(loaded_gcp['title'], 'Cloud Storage JSON API')
test_eq(loaded_gcp['operations'], 1)
gcp_row = gcp.api_ops(name='gcp')[0]
test_eq((gcp_row['group'], gcp_row['name'], gcp_row['path'], gcp_row['required']),
        ('buckets', 'get', 'b/{bucket}', ['bucket']))
test_eq(gcp.specs['gcp'].base_url, 'https://storage.googleapis.com/storage/v1/')
from fastspec.oapi import OpenAPIClient
assert isinstance(OpenAPIClient(gcp.specs['gcp'], sync=True), OpenAPIClient)

### A key and its specification

`headers` accompanies every call. Credentials do not. One host can hold several specifications, and a global `Authorization` header could reach the wrong service. `api_creds` stores credentials by specification. `api_keyed` exposes header names without exposing values.

In [ ]:
RATES = {'openapi': '3.0.0', 'info': {'title': 'Rates', 'version': '1'},
         'servers': [{'url': 'https://api.example.com'}],
         'paths': {'/v1/latest': {'get': {'operationId': 'latest', 'summary': 'Latest rates'}}}}

k = SpecHost(roots=['.'], headers={'User-Agent': 'ramabana'})
k.api_load(RATES, 'rates'); k.api_load(SPEC, 'widgets')
k.api_creds('rates', {'X-Api-Key': 'secret'})
test_eq(k.api_keyed(), {'rates': ['X-Api-Key']})

# the key reaches the spec it was set on, over the headers every call carries, and no further
test_eq(k._client('rates', k.specs['rates']).transport.base_headers,
        {'User-Agent': 'ramabana', 'X-Api-Key': 'secret'})
test_eq(k._client('widgets', k.specs['widgets']).transport.base_headers, {'User-Agent': 'ramabana'})

# fastspec names a group after a path segment. `/v1/latest` leaves `client.latest` a group
# rather than the operation. A group is not callable, which is how `api_call` tells them apart.
assert not callable(getattr(k._client('rates', k.specs['rates']), 'latest'))
test_fail(lambda: k.api_call('nope', name='rates'), contains='no operation')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()